# 04 — RDF Knowledge Graph


In [1]:
# ============================================================
# 04 — RDF KNOWLEDGE GRAPH CONSTRUCTION
# Smart City Knowledge Graph & Network Accessibility Analyzer
# ============================================================

from pathlib import Path
import sys
import geopandas as gpd

# ------------------------------------------------------------
# 1. Resolve project root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(
        f"Could not locate project root.\n"
        f"Current location: {PROJECT_ROOT}\n"
        f"Expected: {PROJECT_ROOT / 'src'}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("=" * 60)
print("RDF KNOWLEDGE GRAPH CONSTRUCTION")
print("=" * 60)
print(f"Project root: {PROJECT_ROOT}")


RDF KNOWLEDGE GRAPH CONSTRUCTION
Project root: /Users/subhankarbiswas/smart-city-knowledge-graph-v3


In [2]:
# ------------------------------------------------------------
# 2. Define input/output paths
# ------------------------------------------------------------

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
GRAPH_DIR = PROJECT_ROOT / "outputs" / "graphs"

GRAPH_DIR.mkdir(parents=True, exist_ok=True)

POIS_WD_PATH = PROCESSED_DIR / "pois_wikidata.parquet"
POIS_CLEAN_PATH = PROCESSED_DIR / "pois_clean.parquet"

BUILDINGS_PATH = PROCESSED_DIR / "buildings_clean.parquet"
ROADS_PATH = PROCESSED_DIR / "roads_clean.parquet"
TRANSPORT_PATH = PROCESSED_DIR / "transport_clean.parquet"

TTL_PATH = GRAPH_DIR / "smart_city_kg.ttl"
JSONLD_PATH = GRAPH_DIR / "smart_city_kg.jsonld"

print("\nInput files:")
for path in [
    POIS_WD_PATH,
    POIS_CLEAN_PATH,
    BUILDINGS_PATH,
    ROADS_PATH,
    TRANSPORT_PATH,
]:
    print(f"  {'✓' if path.exists() else '✗'} {path}")


Input files:
  ✓ /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/pois_wikidata.parquet
  ✓ /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/pois_clean.parquet
  ✓ /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/buildings_clean.parquet
  ✓ /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/roads_clean.parquet
  ✓ /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/transport_clean.parquet


In [3]:
# ------------------------------------------------------------
# 3. Validate required datasets
# ------------------------------------------------------------

required_files = [
    BUILDINGS_PATH,
    ROADS_PATH,
    TRANSPORT_PATH,
]

missing = [
    str(path)
    for path in required_files
    if not path.exists()
]

if not POIS_WD_PATH.exists() and not POIS_CLEAN_PATH.exists():
    missing.append(
        f"{POIS_WD_PATH} OR {POIS_CLEAN_PATH}"
    )

if missing:
    raise FileNotFoundError(
        "The following required datasets are missing:\n\n"
        + "\n".join(f"  - {item}" for item in missing)
        + "\n\nRun the previous notebooks first."
    )

print("\n✓ All required datasets are available.")


✓ All required datasets are available.


In [4]:
# ------------------------------------------------------------
# 4. Load datasets
# ------------------------------------------------------------

# Prefer Wikidata-enriched POIs
if POIS_WD_PATH.exists():
    pois_path = POIS_WD_PATH
    pois = gpd.read_parquet(POIS_WD_PATH)
    poi_source = "OSM + Wikidata"
else:
    pois_path = POIS_CLEAN_PATH
    pois = gpd.read_parquet(POIS_CLEAN_PATH)
    poi_source = "OSM only"

buildings = gpd.read_parquet(BUILDINGS_PATH)
roads = gpd.read_parquet(ROADS_PATH)
transport = gpd.read_parquet(TRANSPORT_PATH)

print("\nDatasets loaded:")
print(f"  POIs:        {len(pois):,} ({poi_source})")
print(f"  Buildings:   {len(buildings):,}")
print(f"  Roads:       {len(roads):,}")
print(f"  Transport:   {len(transport):,}")


Datasets loaded:
  POIs:        225 (OSM + Wikidata)
  Buildings:   10,347
  Roads:       2,355
  Transport:   165


In [5]:
# ------------------------------------------------------------
# 5. Validate geometry and CRS
# ------------------------------------------------------------

datasets = {
    "POIs": pois,
    "Buildings": buildings,
    "Roads": roads,
    "Transport": transport,
}

print("\nGeospatial validation:")
print("-" * 60)

for name, gdf in datasets.items():

    empty_geometry = gdf.geometry.isna().sum()
    invalid_geometry = (~gdf.geometry.is_valid).sum()

    print(f"\n{name}")
    print(f"  Rows:              {len(gdf):,}")
    print(f"  CRS:               {gdf.crs}")
    print(f"  Missing geometry:  {empty_geometry:,}")
    print(f"  Invalid geometry:  {invalid_geometry:,}")


Geospatial validation:
------------------------------------------------------------

POIs
  Rows:              225
  CRS:               {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north"

In [6]:
# ------------------------------------------------------------
# 6. Remove records without geometry
# ------------------------------------------------------------

def prepare_geodataframe(gdf, name):
    gdf = gdf.copy()

    before = len(gdf)

    gdf = gdf[
        gdf.geometry.notna()
    ].copy()

    removed = before - len(gdf)

    if removed:
        print(
            f"{name}: removed {removed:,} "
            "records without geometry"
        )

    return gdf


pois = prepare_geodataframe(pois, "POIs")
buildings = prepare_geodataframe(buildings, "Buildings")
roads = prepare_geodataframe(roads, "Roads")
transport = prepare_geodataframe(transport, "Transport")

print("\n✓ Geospatial datasets prepared.")


✓ Geospatial datasets prepared.


In [7]:
# ------------------------------------------------------------
# 7. Import RDF knowledge graph functions
# ------------------------------------------------------------

from src.rdf_kg import (
    build_knowledge_graph,
    save_graph,
)

print("✓ RDF knowledge graph module loaded.")

✓ RDF knowledge graph module loaded.


In [8]:
# ------------------------------------------------------------
# 8. Build RDF knowledge graph
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("BUILDING KNOWLEDGE GRAPH")
print("=" * 60)

g = build_knowledge_graph(
    pois=pois,
    buildings=buildings,
    roads=roads,
    transport=transport,
)

print("\n✓ Knowledge graph constructed.")
print(f"Total RDF triples: {len(g):,}")


BUILDING KNOWLEDGE GRAPH

✓ Knowledge graph constructed.
Total RDF triples: 49,930


In [9]:
# ------------------------------------------------------------
# 9. Basic RDF graph statistics
# ------------------------------------------------------------

subjects = set()
predicates = set()
objects = set()

for subject, predicate, obj in g:

    subjects.add(subject)
    predicates.add(predicate)
    objects.add(obj)

print("\nRDF statistics:")
print("-" * 60)
print(f"Triples:           {len(g):,}")
print(f"Unique subjects:   {len(subjects):,}")
print(f"Unique predicates: {len(predicates):,}")
print(f"Unique objects:    {len(objects):,}")


RDF statistics:
------------------------------------------------------------
Triples:           49,930
Unique subjects:   23,371
Unique predicates: 7
Unique objects:    25,219


In [10]:
# ------------------------------------------------------------
# 10. Inspect RDF predicates
# ------------------------------------------------------------

from collections import Counter

predicate_counts = Counter(
    str(predicate)
    for _, predicate, _ in g
)

print("\nTop RDF predicates:")
print("-" * 60)

for predicate, count in predicate_counts.most_common(20):
    print(f"{count:>8,}  {predicate}")


Top RDF predicates:
------------------------------------------------------------
  23,366  http://www.w3.org/1999/02/22-rdf-syntax-ns#type
  13,091  http://www.opengis.net/ont/geosparql#asWKT
  11,683  http://www.opengis.net/ont/geosparql#hasGeometry
   1,396  http://www.w3.org/2000/01/rdf-schema#label
     224  https://example.org/smartcity/serviceType
     165  https://example.org/smartcity/transportType
       5  http://www.w3.org/2002/07/owl#sameAs


In [11]:
# ------------------------------------------------------------
# 11. Check Wikidata links
# ------------------------------------------------------------

wikidata_predicates = [
    str(predicate)
    for _, predicate, _ in g
    if "sameAs" in str(predicate)
    or "wikidata" in str(predicate).lower()
]

print("\nWikidata-related RDF statements:")
print(f"  {len(wikidata_predicates):,}")

if wikidata_predicates:
    print("✓ Wikidata reconciliation detected.")
else:
    print(
        "⚠ No Wikidata relationships detected. "
        "The graph may contain OSM-only entities."
    )


Wikidata-related RDF statements:
  5
✓ Wikidata reconciliation detected.


In [12]:
# ------------------------------------------------------------
# 12. Save RDF graph
# ------------------------------------------------------------

print("\nSaving graph...")

save_graph(
    g,
    str(TTL_PATH),
    str(JSONLD_PATH),
)

print("\n✓ Graph saved successfully.")


Saving graph...

✓ Graph saved successfully.


In [13]:
# ------------------------------------------------------------
# 13. Verify output files
# ------------------------------------------------------------

print("\nGenerated files:")
print("-" * 60)

for path in [TTL_PATH, JSONLD_PATH]:

    if path.exists():

        size_kb = path.stat().st_size / 1024

        print(
            f"✓ {path.name:<30} "
            f"{size_kb:,.1f} KB"
        )

    else:

        print(f"✗ Missing: {path}")


Generated files:
------------------------------------------------------------
✓ smart_city_kg.ttl              4,624.0 KB
✓ smart_city_kg.jsonld           9,179.4 KB


In [14]:
# ------------------------------------------------------------
# 14. Final knowledge graph summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("KNOWLEDGE GRAPH COMPLETE")
print("=" * 60)

print(f"""
Source datasets
---------------
POIs:         {len(pois):,}
Buildings:    {len(buildings):,}
Roads:        {len(roads):,}
Transport:    {len(transport):,}
POI source:   {poi_source}

RDF graph
---------
Triples:      {len(g):,}
Subjects:     {len(subjects):,}
Predicates:   {len(predicates):,}
Objects:      {len(objects):,}

Outputs
-------
Turtle:       {TTL_PATH}
JSON-LD:      {JSONLD_PATH}
""")

print("✓ Notebook 04 completed successfully.")


KNOWLEDGE GRAPH COMPLETE

Source datasets
---------------
POIs:         225
Buildings:    10,347
Roads:        2,355
Transport:    165
POI source:   OSM + Wikidata

RDF graph
---------
Triples:      49,930
Subjects:     23,371
Predicates:   7
Objects:      25,219

Outputs
-------
Turtle:       /Users/subhankarbiswas/smart-city-knowledge-graph-v3/outputs/graphs/smart_city_kg.ttl
JSON-LD:      /Users/subhankarbiswas/smart-city-knowledge-graph-v3/outputs/graphs/smart_city_kg.jsonld

✓ Notebook 04 completed successfully.
